# Twinkle 采样推理教程

本教程手把手教你如何使用**已训练的 LoRA 检查点**进行文本生成。

## 你会学到什么

- 如何连接 Twinkle 服务端
- 如何加载训练好的 LoRA 权重
- 如何编码聊天对话并发送采样请求
- 如何解码和查看生成结果

## 前置条件

| 条件 | 说明 |
|------|------|
| Twinkle 服务端 | 必须已启动，参见 `cookbook/client/server/` 下的配置 |
| MODELSCOPE_TOKEN | 在启动 Jupyter 前执行 `export MODELSCOPE_TOKEN=<你的Token>` |
| LoRA 检查点 | 需要一个训练好的检查点路径（`twinkle://` URI） |

> 检查点路径在训练时由 `save_state()` 返回，详见 `self_cognition.ipynb` 或 `short_math_grpo.ipynb`。

## 安装依赖

In [ ]:
# !pip install tinker twinkle

## 1. 初始化客户端

`init_tinker_client()` 会注入 Twinkle 的兼容层。

⚠️ **必须在** `from tinker import ServiceClient` **之前**调用，否则会导入到原版 Tinker 而非 Twinkle 实现。

In [ ]:
import os
from tinker import types

from twinkle.data_format import Message, Trajectory
from twinkle.template import Template
from twinkle import init_tinker_client

# 必须在导入 ServiceClient 之前调用
init_tinker_client()

## 2. 连接服务端

通过 `ServiceClient` 连接 Twinkle 服务。

- `base_url`：Twinkle 服务地址。ModelScope 托管环境使用 `http://www.modelscope.cn/twinkle`，自建服务填写实际地址。
- `api_key`：你的 ModelScope Token，用于身份认证和资源隔离。

In [ ]:
from tinker import ServiceClient

base_model = 'Qwen/Qwen3.5-27B'
base_url = 'http://www.modelscope.cn/twinkle'

service_client = ServiceClient(
    base_url=base_url,
    api_key=os.environ.get('MODELSCOPE_TOKEN')
)
print(f'已连接服务端: {base_url}')

## 3. 加载 LoRA 检查点

用训练时保存的检查点路径创建**采样客户端**。

| 参数 | 说明 |
|------|------|
| `model_path` | `twinkle://` URI，训练时 `save_state()` 返回的路径 |
| `base_model` | 训练时使用的基座模型（必须与训练时一致） |

服务端收到请求后，会加载基座模型并叠加 LoRA 适配器权重。

> 请将下方 `model_path` 替换为你的实际检查点路径。

In [ ]:
# 替换为你的实际检查点路径
model_path = 'twinkle://xxx-Qwen_Qwen3.5-27B-xxx/weights/twinkle-lora-1'

sampling_client = service_client.create_sampling_client(
    model_path=model_path,
    base_model=base_model
)
print(f'采样客户端已就绪，基座模型: {base_model}')

## 4. 构造聊天 Prompt

`Template` 会加载模型对应的 tokenizer 和聊天模板，将多轮对话编码为 token 序列。

关键概念：
- **`Trajectory`**：一次对话，由多条 `Message` 组成（system / user / assistant）
- **`add_generation_prompt=True`**：在对话末尾追加 assistant 角色前缀，提示模型开始生成
- 编码后的 `input_ids` 就是模型实际接收的 token 序列

In [ ]:
template = Template(model_id=f'ms://{base_model}')

# 构造一段对话
trajectory = Trajectory(
    messages=[
        Message(role='system', content='You are a helpful assistant'),
        Message(role='user', content='Who are you?'),
    ]
)

# 编码为 token 序列
input_feature = template.encode(trajectory, add_generation_prompt=True)
input_ids = input_feature['input_ids'].tolist()

print(f'Prompt 共 {len(input_ids)} 个 token')

## 5. 发送采样请求并查看结果

### 采样参数说明

| 参数 | 说明 |
|------|------|
| `max_tokens` | 最大生成 token 数 |
| `temperature` | 随机性控制。0.0 = 贪婪（确定性），越大越随机 |
| `stop` | 遇到这些字符串时提前停止生成 |
| `num_samples` | 对同一 prompt 独立生成的补全条数 |

### 运行效果

如果加载的是自我认知训练的检查点，模型会以自定义身份回答。
如果加载的是数学 GRPO 检查点，可以换一个数学问题来测试。

In [ ]:
prompt = types.ModelInput.from_ints(input_ids)
params = types.SamplingParams(
    max_tokens=128,       # 最多生成 128 个 token
    temperature=0.7,      # 适度随机
    stop=['\n']           # 遇到换行停止
)

print('正在采样...')
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=1)
result = future.result()

print('\n生成结果:')
for i, seq in enumerate(result.sequences):
    decoded = template.decode(seq.tokens)
    print(f'  [{i}] {decoded}')

## 常见问题

| 问题 | 解决方案 |
|------|----------|
| `ConnectionError` | 确认服务端已启动，`base_url` 地址正确 |
| `AuthenticationError` | 检查 `MODELSCOPE_TOKEN` 是否已设置 |
| 输出为空或乱码 | 确认 `base_model` 与训练时一致，`model_path` 路径正确 |
| 响应太慢 | 首次请求需要加载模型，后续请求会快很多 |